# KPI del partido de la noche — NBA
Para una fecha dada, calcula qué partido de la NBA fue el más interesante de ver (nota 0–100), combinando igualdad del marcador, prórrogas, ritmo anotador y mejor actuación individual (Game Score de Hollinger).

## 1. Instalación y descarga de datos
Instalamos `nba_api` y bajamos los partidos de la fecha desde la API oficial de la NBA.

In [1]:
!pip install nba_api

In [2]:
from datetime import date, timedelta

# --- Fecha a analizar ---
# Para la temporada (automático, ayer):
fecha = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

# Para probar ahora (una fecha con partidos):
#fecha = "2026-04-10"

print("Analizando los partidos del:", fecha)

Analizando los partidos del: 2026-09-16


In [3]:
from nba_api.stats.endpoints import scoreboardv3

sb = scoreboardv3.ScoreboardV3(game_date=fecha, timeout=120)
dfs = sb.get_data_frames()
print("tablas:", len(dfs))
for i, d in enumerate(dfs):
    print(i, d.shape, list(d.columns))

tablas: 6
0 (1, 3) ['gameDate', 'leagueId', 'leagueName']
1 (0, 18) ['gameId', 'gameCode', 'gameStatus', 'gameStatusText', 'period', 'gameClock', 'gameTimeUTC', 'gameEt', 'regulationPeriods', 'seriesGameNumber', 'gameLabel', 'gameSubLabel', 'seriesText', 'ifNecessary', 'seriesConference', 'poRoundDesc', 'gameSubtype', 'isNeutral']
2 (0, 12) ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'wins', 'losses', 'score', 'seed', 'inBonus', 'timeoutsRemaining']
3 (0, 12) ['gameId', 'teamId', 'leaderType', 'personId', 'name', 'playerSlug', 'jerseyNum', 'position', 'teamTricode', 'points', 'rebounds', 'assists']
4 (0, 13) ['gameId', 'teamId', 'leaderType', 'personId', 'name', 'playerSlug', 'jerseyNum', 'position', 'teamTricode', 'points', 'rebounds', 'assists', 'seasonLeadersFlag']
5 (0, 6) ['gameId', 'broadcasterType', 'broadcasterId', 'broadcastDisplay', 'broadcasterTeamId', 'broadcasterDescription']


## 2. Motor del KPI
Las funciones que convierten cada métrica en nota (0–1) y los pesos.

In [4]:
"""
KPI v1 "Partido del dia" - motor de puntuacion (solo box score).
Escala 0-100. Cada componente se normaliza a 0-1 y se pondera.
Los PESOS son una hipotesis: se calibran mirando noches reales.
"""

def _clip01(x):
    return max(0.0, min(1.0, x))

# ---- Componentes (cada uno devuelve 0..1) ----

def closeness_score(final_margin):
    """Igualdad del resultado. Margen 0 -> 1.0 ; margen >=20 -> 0."""
    return _clip01(1 - final_margin / 20.0)

def overtime_score(num_ot):
    """Prorrogas. 0 -> 0 ; 1 -> 0.7 ; 2+ -> 1.0."""
    if num_ot <= 0:
        return 0.0 
    return _clip01(0.7 + 0.3 * (num_ot - 1))

def pace_score(total_points):
    """Ritmo anotador. 200 pts totales -> 0 ; 260 -> 1 (rango tipico NBA)."""
    return _clip01((total_points - 200) / 60.0)

def star_score(best_game_score):
    """Mejor actuacion individual (game score de Hollinger). 15 -> 0 ; 45 -> 1."""
    return _clip01((best_game_score - 15) / 30.0)

# ---- Pesos v1 (suman 1.0) ----
WEIGHTS = {"closeness": 0.45, "overtime": 0.20, "pace": 0.05, "star": 0.30}


## 3. Marcador de cada partido
Sacamos margen final y puntos totales de cada partido.

In [5]:
line = dfs[2]

# Agrupamos por partido y, sobre el marcador de cada grupo, sacamos varias cosas
resumen = line.groupby("gameId")["score"].agg(
    total="sum",
    maximo="max",
    minimo="min"
)
resumen["margen"] = resumen["maximo"] - resumen["minimo"]
if len(resumen) == 0:
    print("No hubo partidos el", fecha)

print(resumen.head())

No hubo partidos el 2026-09-16
Empty DataFrame
Columns: [total, maximo, minimo, margen]
Index: []


## 4. Prórrogas
Contamos las prórrogas de cada partido y las añadimos a `resumen`.

In [6]:
# Recuperamos la cabecera del 10 de abril y sacamos las prorrogas de la columna correcta
header = dfs[1].copy()
header["prorrogas"] = header["period"] - 4

# Lo unimos a nuestro 'resumen' (que tenia total y margen) por el gameId
resumen = resumen.join(header.set_index("gameId")["prorrogas"])

print(resumen[["margen", "total", "prorrogas"]])

Empty DataFrame
Columns: [margen, total, prorrogas]
Index: []


## 5. La estrella de cada partido
Bajamos el box score de cada partido y nos quedamos con la mejor actuación (Game Score).

In [7]:
import time
from nba_api.stats.endpoints import boxscoretraditionalv3

def game_score_col(jug):
    return (jug["points"]
        + 0.4*jug["fieldGoalsMade"] - 0.7*jug["fieldGoalsAttempted"]
        - 0.4*(jug["freeThrowsAttempted"] - jug["freeThrowsMade"])
        + 0.7*jug["reboundsOffensive"] + 0.3*jug["reboundsDefensive"]
        + jug["steals"] + 0.7*jug["assists"] + 0.7*jug["blocks"]
        - 0.4*jug["foulsPersonal"] - jug["turnovers"])

estrellas = {}          # gameId -> mejor game score
estrella_nombre = {}    # gameId -> nombre del mejor jugador

for gid in resumen.index:
    bx = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=gid, timeout=120)
    jug = bx.get_data_frames()[0]
    jug["gs"] = game_score_col(jug)

    mejor = jug.loc[jug["gs"].idxmax()]          # <-- la fila del mejor jugador
    estrellas[gid] = mejor["gs"]
    estrella_nombre[gid] = f'{mejor["firstName"]} {mejor["familyName"]}'  # <-- su nombre

    print(f'{gid}: {estrella_nombre[gid]} ({round(estrellas[gid],1)})')
    time.sleep(2)

print("\nListo, procesados", len(estrellas), "partidos")


Listo, procesados 0 partidos


## 6. Calcular el KPI
Combinamos las cuatro notas con sus pesos para el KPI (0–100) de cada partido.

In [8]:
# --- Añadir enfrentamiento (equipos) y nombre de la estrella al resumen ---
line = dfs[2]
enfrentamientos = line.groupby("gameId")["teamTricode"].apply(lambda x: " vs ".join(x))
resumen["partido"]  = resumen.index.map(enfrentamientos)
resumen["estrella"] = resumen.index.map(estrellas)
resumen["quien"]    = resumen.index.map(estrella_nombre)

# --- Las cuatro notas y el KPI ---
resumen["nota_igualdad"] = resumen["margen"].apply(closeness_score)
resumen["nota_ritmo"]    = resumen["total"].apply(pace_score)
resumen["nota_ot"]       = resumen["prorrogas"].apply(overtime_score)
resumen["nota_estrella"] = resumen["estrella"].apply(star_score)

W = WEIGHTS
resumen["KPI"] = round(100 * (
      W["closeness"]*resumen["nota_igualdad"]
    + W["overtime"] *resumen["nota_ot"]
    + W["pace"]     *resumen["nota_ritmo"]
    + W["star"]     *resumen["nota_estrella"]
), 1)

## 7. Ranking final
Ordenamos por KPI. La primera tabla es la limpia (presentar); la segunda muestra el desglose de notas (analizar).

In [9]:
# --- Ranking final, presentable ---
if len(resumen) == 0:
    print("No hubo partidos el", fecha)
else:
    ranking = resumen.sort_values("KPI", ascending=False).reset_index(drop=True)
    ranking.index = ranking.index + 1

    tabla = ranking[["partido", "quien", "estrella", "margen", "total", "prorrogas", "KPI"]].rename(columns={
        "partido":   "Partido",
        "quien":     "Estrella",
        "estrella":  "Game Score",
        "margen":    "Margen",
        "total":     "Puntos",
        "prorrogas": "Prórrogas",
    })

    mejor = tabla.iloc[0]
    print(f"Partido de la noche: {mejor['Partido']}  ·  {mejor['Estrella']} ({mejor['Game Score']})  ·  KPI {mejor['KPI']}")
    print("-" * 70)
    print(tabla.to_string())

No hubo partidos el 2026-09-16


In [10]:
# --- Ranking final con desglose de notas ---
if len(resumen) == 0:
    print("No hubo partidos el", fecha)
else:
    ranking = resumen.sort_values("KPI", ascending=False).reset_index(drop=True)
    ranking.index = ranking.index + 1

    tabla = ranking[[
        "partido", "quien",
        "margen", "nota_igualdad",
        "total", "nota_ritmo",
        "prorrogas", "nota_ot",
        "estrella", "nota_estrella",
        "KPI"
    ]].rename(columns={
        "partido":       "Partido",
        "quien":         "Estrella",
        "margen":        "Margen",
        "nota_igualdad": "N.Igualdad",
        "total":         "Puntos",
        "nota_ritmo":    "N.Ritmo",
        "prorrogas":     "Prórrogas",
        "nota_ot":       "N.OT",
        "estrella":      "GameScore",
        "nota_estrella": "N.Estrella",
    }).round(2)

    mejor = tabla.iloc[0]
    print(f"Partido de la noche: {mejor['Partido']}  ·  {mejor['Estrella']} (GS {mejor['GameScore']})  ·  KPI {mejor['KPI']}")
    print("-" * 100)
    print(tabla.to_string())

No hubo partidos el 2026-09-16
